Data cleaning and Processing.
1.Converting date variables into appropriate data types.
2.Identifying invalid or inconsistent numerical values.
3.Examining categorical variables for inconsistencies.
4.Prepairing a reliable dataset for subsequent analysis.



In [1]:
# importing libraries for data cleaning and manipulation
import pandas as pd
import numpy as np

In [2]:
#importing the dataset 
df = pd.read_csv("../data/fmcg_sales_3years_1M_rows.csv")

In [3]:
#check the shape of the dataset
df.shape

(1100000, 33)

In [4]:
#check the current data type of the date column
df['date'].dtype

<StringDtype(na_value=nan)>

In [5]:
#converting the date column from  string to datetime
df['date'] = pd.to_datetime(df['date'])

In [6]:
#verify the data type of the datatype
df['date'].dtype

dtype('<M8[us]')

In [7]:
#check the earliest and latest dates in the dataset
print("Earliest date:", df['date'].min())
print("Latest date:", df['date'].max())

Earliest date: 2021-01-01 00:00:00
Latest date: 2023-12-31 00:00:00


In [8]:
#counting missing values in the date column
df['date'].isna().sum()

np.int64(0)

In [9]:
#check for year mismatches
(df['date'].dt.year != df['year']).sum()

np.int64(0)

In [10]:
#checking whether the day column matches the date column
(df['date'].dt.day != df['day']).sum()

np.int64(0)

In [11]:
#identifying the numercal columns in the dataset
df.select_dtypes(include='number').columns.tolist()

['year',
 'month',
 'day',
 'weekofyear',
 'weekday',
 'is_weekend',
 'is_holiday',
 'temperature',
 'rain_mm',
 'latitude',
 'longitude',
 'units_sold',
 'list_price',
 'discount_pct',
 'promo_flag',
 'gross_sales',
 'net_sales',
 'stock_on_hand',
 'stock_out_flag',
 'lead_time_days',
 'purchase_cost',
 'margin_pct']

In [12]:
# selecting numerical variables that should not be negative
columns_to_check = [
    'units_sold',
    'list_price',
    'stock_on_hand',
    'purchase_cost',
    'lead_time_days'
]
#count negative values in each selected column
(df[columns_to_check] < 0).sum()

units_sold        0
list_price        0
stock_on_hand     0
purchase_cost     0
lead_time_days    0
dtype: int64

This validation identified zero values across all variables.Futher validation is required to investigate zero values,percentage ranges and potential outliers.

In [13]:
# examining the minimum and maximum values of percentage variables
df[['discount_pct', 'margin_pct']].agg(['min', 'max'])

,discount_pct,margin_pct
min,0.0,-0.05
max,0.3,0.55


The discount percentages ranged from 0-30% ,indicating that discounts were recorded as decimal proprtions and were plausable range.
The profit margin ranged from -5-55% ,negative values may represent products sold at a loss and not necessarily data entry errors.

In [14]:
# count zero values in selected numericl variables
(df[columns_to_check] == 0).sum()

units_sold        3092
list_price           0
stock_on_hand      108
purchase_cost        0
lead_time_days       0
dtype: int64

The analysis identified 3,092 observations with zero units sold, representing approximately 0.28% of the dataset.
Additionally, 108 observations recorded zero stock on hand, representing approximately 0.01% of the dataset.
Zero sales and aero inventory could be caused by legitimate business problems so futher investigation is required.

In [15]:
# Count observations where both units sold and stock on hand are zero
((df['units_sold'] == 0) & (df['stock_on_hand'] == 0)).sum()

np.int64(0)

All 3,092 observations with zero sales had positive recorded inventory levels, while the 108 observations with zero inventory recorded positive sales.
This suggests that zero sales were not directly associated with zero recorded inventory in the same observations.
However, the interpretation depends on how inventory levels were measured. For example, zero stock may represent inventory remaining after sales occurred.Futher investigation is required before drawing conclusions on inventory constraints and customer demand.

In [16]:
# Count the number of observations for each stockout flag value
df['stock_out_flag'].value_counts(dropna=False)

stock_out_flag
0    1066886
1      33114
Name: count, dtype: int64

A total of 1,066,886 observations (96.99%) were classified as having no stockout, while 33,114 observations (3.01%) were flagged as stockouts.
However, only 108 observations recorded zero stock on hand.
This difference suggests that the stockout indicator and recorded inventory levels may represent different aspects of inventory availability.
Further investigation is necessary to understand their relationship before using these variables in demand forecasting.

In [17]:
# Compare stockout flags with whether inventory is zero
pd.crosstab(
    df['stock_out_flag'],
    df['stock_on_hand'] == 0
)

stock_on_hand,False,True
stock_out_flag,,
0,1066782,104
1,33110,4


The analysis revealed that 33,110 observations were flagged as stockouts despite having positive recorded inventory levels.
Additionally, 104 observations recorded zero inventory without being flagged as stockouts, while only four observations had both zero inventory and a stockout flag.
These findings indicate that the stockout indicator does not correspond directly to zero recorded inventory.
This discrepancy may reflect differences in how inventory availability and stockout events are measured. However, the precise explanation cannot be established from the available results.
Both variables are retained for further analysis, and the discrepancy will be considered when interpreting inventory constraints and forecasting demand.

In [18]:
#check the unique values in the remaining binary indicators 
binary_columns = ['is_weekend', 'is_holiday', 'promo_flag']
for col in binary_columns:
    print(col, df[col].unique())

is_weekend [0 1]
is_holiday [1 0]
promo_flag [1 0]


The `is_weekend`, `is_holiday`, and `promo_flag` variables were examined to verify that they contained valid binary values.
All three variables contained only 0 and 1, indicating that no unexpected values were detected.
The `stock_out_flag` variable was also previously confirmed to contain only 0 and 1.Therefore, no corrections were required for the binary indicators based on their observed values.


In [19]:
# Identify categorical columns in the dataset
categorical_columns = df.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

categorical_columns

['store_id',
 'country',
 'city',
 'channel',
 'sku_id',
 'sku_name',
 'category',
 'subcategory',
 'brand',
 'supplier_id']

In [20]:
# Select categorical variables for consistency checks
category_columns = [
    'country',
    'city',
    'channel',
    'category',
    'subcategory',
    'brand'
]
# Count the number of unique values in each column
df[category_columns].nunique()

country         7
city            9
channel         4
category        5
subcategory    17
brand           6
dtype: int64

In [21]:
# Display unique values for selected categorical variables
for col in ['country', 'channel', 'category']:
    print(f"\n{col}:")
    print(df[col].unique())


country:
<ArrowStringArray>
['Germany', 'Italy', 'Poland', 'France', 'Spain', 'Austria', 'Netherlands']
Length: 7, dtype: str

channel:
<ArrowStringArray>
['Hypermarket', 'Supermarket', 'E-commerce', 'Convenience']
Length: 4, dtype: str

category:
<ArrowStringArray>
['Personal Care', 'Beverages', 'Dairy', 'Snacks', 'Home Care']
Length: 5, dtype: str


In [22]:
# Check the remaining categorical labels
for col in ['city', 'subcategory', 'brand']:
    print(f"\n{col}:")
    print(df[col].unique())


city:
<ArrowStringArray>
[   'Berlin',      'Rome',     'Milan',    'Warsaw',     'Paris', 'Barcelona',
    'Vienna',    'Madrid', 'Amsterdam']
Length: 9, dtype: str

subcategory:
<ArrowStringArray>
[     'Shampoo',        'Water',       'Yogurt',    'Chocolate',
         'Soda',    'Detergent',       'Cheese',        'Juice',
         'Soap',         'Milk', 'Energy drink',        'Chips',
     'Biscuits',      'Cleaner',     'Softener',         'Nuts',
   'Toothpaste']
Length: 17, dtype: str

brand:
<ArrowStringArray>
['BrandB', 'BrandD', 'BrandA', 'BrandF', 'BrandC', 'BrandE']
Length: 6, dtype: str


Inspection of all six categorical variables revealed no obvious inconsistencies in spelling or capitalization. The category labels appeared consistent, and no standardization changes were required based on the observed values.


In [24]:
# Save the prepared dataset
df.to_csv("../data/fmcg_sales_cleaned.csv", index=False)
print("Prepared dataset saved successfully!")

Prepared dataset saved successfully!
